# 11. Ders: Algoritmik Empati (MoodJar - Bölüm 3)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Geçen haftalarda yapay zekanın kalbini oluşturduk ve Türkçe anlamasını sağladık. Ancak bir projenin hayata geçebilmesi için sadece 'anlamak' yetmez, kullanıcının o uygulamayla doğal bir şekilde etkileşime girmesi gerekir. Bir kafede sipariş verirken barista siparişinizi aldıktan sonra size arkanı dönüp sessizce beklemez; "Tamamdır, hemen hazırlıyorum" diyerek bir etkileşim döngüsü (feedback) oluşturur.

Bilgisayar bilimlerinde buna İnsan-Makine Etkileşimi (HCI) diyoruz. Yapay zekanın bize sadece 'Sen %90 ihtimalle mutsuzsun' demesi sinir bozucudur. Bunun yerine 'Anlıyorum, zor bir gün geçiriyorsun, konuşmak ister misin?' demesi ise 'Algoritmik Empati'dir. Bu derste programlamayı ve yapay zekayı birleştirerek, 7/24 uyanık kalan ve bizimle empati kuran tam teşekküllü bir MoodJar asistanı kodlayacağız.

## Adım 2: Teknik Terimler ve Sözlük

- Algorithmic Empathy (Algoritmik Empati): Makinenin kullanıcının duygusal durumuna uygun, insan benzeri şefkatli veya motive edici yanıtlar üretebilme becerisi.
- UX (User Experience - Kullanıcı Deneyimi): Uygulamanın kullanım kolaylığı ve kullanıcıda bıraktığı his.
- Infinite Loop (Sonsuz Döngü): Kodun kendi kendine durmadığı, sadece belirli bir dış komutla (örneğin kullanıcının 'Çıkış' yazmasıyla) durdurulabilen `while True` döngüleri.
- Fallback (Geri Çekilme Senaryosu): Modelin cümleyi anlayamadığı veya emin olmadığı (düşük skor) durumlarda sistemin çökmemesi için yazılan B planı kodudur.
- Random (Rastgelelik): Uygulamanın sıkıcı olmaması için aynı duruma farklı kelimelerle cevap verebilmesini sağlayan kütüphanedir.

## Adım 3: Random Kütüphanesi ile Rastgele (Tekdüze Olmayan) Cevaplar Üretme

Eğer botumuz her üzgün olduğumuzda "Üzülme geçer" derse bir robot olduğunu çok belli eder. Buna rastgelelik katalım.

In [ ]:
import random

# Negatif durumlar için bir cevap havuzu oluşturuyoruz
negatif_cevaplar = [
    "Görünüşe göre canını sıkan bir şeyler var. Derin bir nefes almayı unutma.",
    "Bugün zor geçmiş olabilir ama yarın yepyeni bir sayfa. Ben buradayım.",
    "Hislerini anlıyorum. Bazen sadece durup dinlenmek en iyisidir."
]

# random.choice() fonksiyonu, bir listenin içinden rastgele bir eleman seçer
secilen_cevap = random.choice(negatif_cevaplar)
print("Asistanın Rastgele Cevabı:", secilen_cevap)

# Bu hücreyi birkaç kez arka arkaya çalıştırarak farklı cevaplar geldiğini gözlemleyin.

## Adım 4: Türkçe NLP Modelimizi ve Kütüphaneleri Dahil Etme

Tam projeye başlamadan önce gerekli araç kitimizi hazırlıyoruz.

In [ ]:
# Not: Eğer Colab ortamınızı yeniden başlattıysanız önce '!pip install transformers' yapmanız gerekebilir.
from transformers import pipeline
import random
import time # Asistanın anında değil, düşünüyormuş gibi beklemesi için.

# Türkçe duygu analizi modelimizi yüklüyoruz.
duygu_modeli = pipeline("sentiment-analysis", model="savasy/bert-base-turkish-sentiment-cased")

print("Modeller ve Kütüphaneler Hazır!")

## Adım 5: Cevap Havuzlarını (Sözlükleri) Oluşturma

Projeyi düzenli tutmak için tüm cevapları bir yerde toplayalım.

In [ ]:
pozitif_cevaplar = [
    "Ne kadar güzel! Bu enerjinin tüm gününe yayılmasını dilerim.",
    "Harika bir haber. Beni de mutlu ettin!",
    "Pozitif hissetmek sana çok yakışıyor, devam et!"
]

negatif_cevaplar = [
    "Görünüşe göre zor bir anındasın. Unutma, her karanlığın sonu aydınlıktır.",
    "Bunu duyduğuma üzüldüm. Derin bir nefes al ve kendine fazla yüklenme.",
    "Anlıyorum. Bazen işler ters gidebilir, önemli olan pes etmemek."
]

kafa_karisikligi_cevaplari = [
    "Hmm, sanırım ne hissettiğinden pek emin değilsin veya biraz karmaşık düşünüyorsun.",
    "Duyguların birbirine girmiş gibi. Biraz daha detaylandırmak ister misin?",
    "Bu cümlenden tam olarak bir duygu çıkaramadım, hayat bazen grilerle doludur."
]

## Adım 6: Sonsuz Döngü ile Ana Asistan Yazılımını Kurma

Şimdi tüm parçaları birleştiriyoruz. `while True` döngüsü ile asistanın 7/24 uyanık kalmasını ve kullanıcının `input()` ile veri girmesini sağlayacağız.

In [ ]:
print("=" * 40)
print("MOODJAR - HİS KAVANOZU UYGULAMASINA HOŞ GELDİNİZ")
print("Bugün kendini nasıl hissediyorsun? Bana içinden geçenleri yaz.")
print("(Çıkmak için 'çıkış' veya 'q' yazabilirsiniz)")
print("=" * 40)

# Güvenlik duvarı (Fallback) eşiği. Modelin %70'ten daha emin olması gerekir.
MINIMUM_SKOR = 0.70 

# Ana Etkileşim Döngüsü
while True:
    # 1. Kullanıcıdan veri al
    kullanici_notu = input("\nSiz: ")
    
    # 2. Çıkış kontrolü yap
    if kullanici_notu.lower() == "çıkış" or kullanici_notu.lower() == "q":
        print("\nMoodJar: Umarım sana biraz da olsa yardımcı olabilmişimdir. Hoşça kal!")
        break # Döngüyü kır ve programı bitir
        
    # Boş bir şey yazarsa uyarı ver
    if len(kullanici_notu.strip()) < 3:
        print("MoodJar: Lütfen bana birkaç kelimeden oluşan anlamlı bir şey yaz.")
        continue # Döngünün başına (while'a) geri dön
        
    # 3. Model analizi (Gerçekçi bir bekleme süresi ekleyelim)
    print("MoodJar düşünüyor...")
    # time.sleep(1) # Bilgisayarın anında cevap vermesi yerine 1 saniye beklemesini sağlar. Colab'de kapalı tutabiliriz.
    
    analiz_sonucu = duygu_modeli(kullanici_notu)
    etiket = analiz_sonucu[0]['label']
    skor = analiz_sonucu[0]['score']
    
    # 4. Algoritmik Empati ve Yanıt Üretme
    if skor < MINIMUM_SKOR:
        # Model emin değilse karmaşık cevap ver
        print("MoodJar:", random.choice(kafa_karisikligi_cevaplari))
        
    elif etiket == "positive": # Modele göre etiketlerin küçük/büyük yazımına dikkat edin
        print("MoodJar:", random.choice(pozitif_cevaplar))
        
    elif etiket == "negative":
        print("MoodJar:", random.choice(negatif_cevaplar))

## Adım 7: Dene ve Keşfet - Asistanı Kandırmaya Çalışmak

Yukarıdaki döngüyü başlattıktan sonra, bota ardı ardına şu tarz cümleler yazarak sistemi test edin:
1. "Bugün dondurma yedim ama midem çok ağrıdı." (Beklenen: Negatif veya Karmaşık)
2. "Sınavdan 10 aldım!" (Eğer model '10' rakamının Türkiye'de kötü olduğunu bilmiyorsa, ünlem işaretinden dolayı pozitif diyebilir. Bu bir model yanılgısıdır / bias.)
3. "Fena değil." (Sıradan bir cevap, modelin kafası karışabilir.)

Asistanınızı yanıltmayı başardığınız durumları not edin. Gerçek hayatta yapay zeka mühendisleri (AI Engineers) sistemi böyle test ederek iyileştirirler.

## Adım 8: Sıra Sende Görevi - Kişiselleştirilmiş MoodJar

Problem: Tasarladığımız MoodJar çalışıyor, ancak biraz 'genel' konuşuyor. Sizden kullanıcının adını öğrenen ve cevaplarında ismini kullanan, aynı zamanda log (kayıt) tutan bir sisteme dönüştürmeniz isteniyor.

Gereksinimler:
1. Yeni bir hücrede `while True` döngüsünü baştan yazın.
2. Döngü BAŞLAMADAN ÖNCE (while'ın dışında) kullanıcıya `isim = input("Adınız nedir?")` diyerek ismini sorun.
3. Döngü BAŞLAMADAN ÖNCE boş bir liste oluşturun: `duygu_kayitlari = []`
4. Döngü içinde (while bloğunda) bot cevap verirken kullanıcının adını da söylesin (Örn: "Anlıyorum Ahmet. Zor bir gün geçmiş.") - İpucu: f-string kullanın.
5. Döngü içinde her analiz yapıldığında, bulunan etiketi ('positive' veya 'negative') `duygu_kayitlari` listesine ekleyin (`.append()`)
6. Kullanıcı 'çıkış' yazdığında, sistemi kapatmadan hemen önce `duygu_kayitlari` listesinin uzunluğunu (len) kontrol edin ve kullanıcıya "Bugün benimle toplam 5 his paylaştın, teşekkürler {isim}" diyerek veda edin.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 11. Dersi tamamlayarak NLP ve Yazılım Mühendisliğini birleştirip ilk tam bağımsız, etkileşimli Yapay Zeka Uygulamanızı yazdınız. 3 derslik 'MoodJar' maratonunu başarıyla bitirdiniz.

Sonraki dersimizde GCA Coding Jam'in bir diğer meşhur projesi olan 'Buzdolabı Şefi' (Fridge Chef) uygulamasına geçiş yapacağız ve metinlerin matematiksel uzayda birbirine ne kadar benzediğini ölçeceğiz.